In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from scipy import stats

# ============================================================
# 1. CARGAR LOS DATASETS
# ============================================================
train = pd.read_csv(r"C:\Users\Ing. Antonio Rial\OneDrive - Universidad Austral\MCD_Laboratorio.de.Implementación.2\Competencia.House.Pricing\data\tabular\train_processed.csv", low_memory=False)

In [2]:
# --- Preprocesamiento train ---
# Seleccionar columnas numéricas (excluyendo IDs y texto)
numeric_cols_train = train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols_train.remove('zpid')  # Eliminar ID

In [3]:
# Eliminar columnas con muchos valores faltantes (ej. más del 30% de NaN)
missing_pct_train = train[numeric_cols_train].isnull().mean()
cols_to_keep_train = missing_pct_train[missing_pct_train < 0.3].index.tolist()

In [4]:
# Reemplazar NaN restantes con la media
train_clean = train[cols_to_keep_train].copy()
train_clean = train_clean.fillna(train_clean.mean())

In [11]:
# Definir variables predictoras (X) y target (y) - log_price
X_train = train_clean.drop(['lastSoldPrice_hpi_adjusted', 'log_price'], axis=1)
y_train_raw = train_clean['lastSoldPrice_hpi_adjusted']
y_train = train_clean['log_price']

In [6]:
# Entrenar modelo
model_raw = LinearRegression()
model_raw.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [7]:
X_train_log, X_val_log, y_train_log, y_val_log = train_test_split(X_train, y_train, test_size=0.2, random_state=42)


In [8]:
model_raw.fit(X_train_log, y_train_log)
y_train_pred = model_raw.predict(X_train_log)

In [9]:
from sklearn.metrics import mean_absolute_error

print("\n📊 === VALIDACIÓN INTERNA (20% de train) ===")
print(f"R²   : {r2_score(y_train, y_train_pred):.4f}")
print("\n📊 === VALIDACIÓN INTERNA (20% de train) ===")
print(f"R²   : {r2_score(y_train, y_train_pred):.4f}")
print(f"MAE  : {mean_absolute_error(y_train, y_train_pred):,.2f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_train, y_train_pred)):,.2f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_train, y_train_pred)):,.2f}")


📊 === VALIDACIÓN INTERNA (20% de train) ===


ValueError: Found input variables with inconsistent numbers of samples: [11840, 9472]

In [ ]:

# --- Modelo 1: Regresión lineal con 'lastSoldPrice_hpi_adjusted' ---
model_raw = LinearRegression()
model_raw.fit(X_train, y_train_pred)
y_train_pred = model_raw.predict(X_test)

rmse_raw = np.sqrt(mean_squared_error(y_test_raw, y_train_pred))
r2_raw = r2_score(y_test_raw, y_train_pred)

# --- Modelo 2: Regresión lineal con 'log_price' ---
model_log = LinearRegression()
model_log.fit(X_train, y_train_log)
y_pred_log = model_log.predict(X_test)

rmse_log = np.sqrt(mean_squared_error(y_test_log, y_pred_log))
r2_log = r2_score(y_test_log, y_pred_log)

# --- Visualización ---
plt.figure(figsize=(14, 6))

# Gráfico 1: Valores reales vs predichos (raw)
plt.subplot(1, 2, 1)
plt.scatter(y_test_raw, y_pred_raw, alpha=0.6)
plt.plot([y_test_raw.min(), y_test_raw.max()], [y_test_raw.min(), y_test_raw.max()], 'r--')
plt.xlabel('Precio Real (USD)')
plt.ylabel('Precio Predicho (USD)')
plt.title(f'Modelo Raw (RMSE: {rmse_raw:.2f}, R²: {r2_raw:.2f})')

# Gráfico 2: Valores reales vs predichos (log)
plt.subplot(1, 2, 2)
plt.scatter(y_test_log, y_pred_log, alpha=0.6)
plt.plot([y_test_log.min(), y_test_log.max()], [y_test_log.min(), y_test_log.max()], 'r--')
plt.xlabel('Log Precio Real')
plt.ylabel('Log Precio Predicho')
plt.title(f'Modelo Log (RMSE: {rmse_log:.2f}, R²: {r2_log:.2f})')

plt.tight_layout()
plt.show()

# --- Resumen ---
print("=== Comparación de Modelos ===")
print(f"Modelo Raw (USD): RMSE={rmse_raw:.2f}, R²={r2_raw:.2f}")
print(f"Modelo Log: RMSE={rmse_log:.2f}, R²={r2_log:.2f}")

In [ ]:
# Residuos para modelo log
residuals = y_test_log - y_pred_log

plt.figure(figsize=(12, 5))

# Histograma de residuos
plt.subplot(1, 2, 1)
sns.histplot(residuals, kde=True)
plt.title('Distribución de Residuos (Log)')

# Q-Q plot
plt.subplot(1, 2, 2)
stats.probplot(residuals, dist="norm", plot=plt)
plt.title('Q-Q Plot (Normalidad)')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

# Cargar y preprocesar datos (igual que antes)
# df = pd.read_csv("train_processed_consistente.csv")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('zpid')
missing_pct = df[numeric_cols].isnull().mean()
cols_to_keep = missing_pct[missing_pct < 0.3].index.tolist()
df_clean = df[cols_to_keep].copy()
df_clean = df_clean.fillna(df_clean.mean())

# Variables predictoras y targets
X = df_clean.drop(['lastSoldPrice_hpi_adjusted', 'log_price'], axis=1)
y_raw = df_clean['lastSoldPrice_hpi_adjusted']
y_log = df_clean['log_price']

# Escalado (importante para regularización)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Dividir en train/test
X_train, X_test, y_train_raw, y_test_raw = train_test_split(X_scaled, y_raw, test_size=0.2, random_state=42)
X_train, X_test, y_train_log, y_test_log = train_test_split(X_scaled, y_log, test_size=0.2, random_state=42)

# --- Diccionario para almacenar resultados ---
results = {
    'Model': [],
    'Target': [],
    'RMSE': [],
    'R²': [],
    'Best Alpha': []
}

# --- Modelo 1: Regresión Lineal (Base) ---
lr_raw = LinearRegression().fit(X_train, y_train_raw)
lr_log = LinearRegression().fit(X_train, y_train_log)

results['Model'].append('Linear')
results['Target'].append('Raw')
results['RMSE'].append(np.sqrt(mean_squared_error(y_test_raw, lr_raw.predict(X_test))))
results['R²'].append(r2_score(y_test_raw, lr_raw.predict(X_test)))
results['Best Alpha'].append('N/A')

results['Model'].append('Linear')
results['Target'].append('Log')
results['RMSE'].append(np.sqrt(mean_squared_error(y_test_log, lr_log.predict(X_test))))
results['R²'].append(r2_score(y_test_log, lr_log.predict(X_test)))
results['Best Alpha'].append('N/A')

# --- Modelo 2: Lasso (L1) ---
lasso_raw = Lasso(random_state=42)
lasso_log = Lasso(random_state=42)
param_grid = {'alpha': [0.001, 0.01, 0.1, 1, 10]}

grid_lasso_raw = GridSearchCV(lasso_raw, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_lasso_raw.fit(X_train, y_train_raw)
best_lasso_raw = grid_lasso_raw.best_estimator_

grid_lasso_log = GridSearchCV(lasso_log, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_lasso_log.fit(X_train, y_train_log)
best_lasso_log = grid_lasso_log.best_estimator_

results['Model'].append('Lasso')
results['Target'].append('Raw')
results['RMSE'].append(np.sqrt(mean_squared_error(y_test_raw, best_lasso_raw.predict(X_test))))
results['R²'].append(r2_score(y_test_raw, best_lasso_raw.predict(X_test)))
results['Best Alpha'].append(grid_lasso_raw.best_params_['alpha'])

results['Model'].append('Lasso')
results['Target'].append('Log')
results['RMSE'].append(np.sqrt(mean_squared_error(y_test_log, best_lasso_log.predict(X_test))))
results['R²'].append(r2_score(y_test_log, best_lasso_log.predict(X_test)))
results['Best Alpha'].append(grid_lasso_log.best_params_['alpha'])

# --- Modelo 3: Ridge (L2) ---
ridge_raw = Ridge(random_state=42)
ridge_log = Ridge(random_state=42)
param_grid = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}

grid_ridge_raw = GridSearchCV(ridge_raw, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_ridge_raw.fit(X_train, y_train_raw)
best_ridge_raw = grid_ridge_raw.best_estimator_

grid_ridge_log = GridSearchCV(ridge_log, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_ridge_log.fit(X_train, y_train_log)
best_ridge_log = grid_ridge_log.best_estimator_

results['Model'].append('Ridge')
results['Target'].append('Raw')
results['RMSE'].append(np.sqrt(mean_squared_error(y_test_raw, best_ridge_raw.predict(X_test))))
results['R²'].append(r2_score(y_test_raw, best_ridge_raw.predict(X_test)))
results['Best Alpha'].append(grid_ridge_raw.best_params_['alpha'])

results['Model'].append('Ridge')
results['Target'].append('Log')
results['RMSE'].append(np.sqrt(mean_squared_error(y_test_log, best_ridge_log.predict(X_test))))
results['R²'].append(r2_score(y_test_log, best_ridge_log.predict(X_test)))
results['Best Alpha'].append(grid_ridge_log.best_params_['alpha'])

# --- Modelo 4: ElasticNet (L1 + L2) -->
elastic_raw = ElasticNet(random_state=42)
elastic_log = ElasticNet(random_state=42)
param_grid = {'alpha': [0.001, 0.01, 0.1, 1], 'l1_ratio': [0.1, 0.5, 0.7, 0.9]}

grid_elastic_raw = GridSearchCV(elastic_raw, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_elastic_raw.fit(X_train, y_train_raw)
best_elastic_raw = grid_elastic_raw.best_estimator_

grid_elastic_log = GridSearchCV(elastic_log, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_elastic_log.fit(X_train, y_train_log)
best_elastic_log = grid_elastic_log.best_estimator_

results['Model'].append('ElasticNet')
results['Target'].append('Raw')
results['RMSE'].append(np.sqrt(mean_squared_error(y_test_raw, best_elastic_raw.predict(X_test))))
results['R²'].append(r2_score(y_test_raw, best_elastic_raw.predict(X_test)))
results['Best Alpha'].append(grid_elastic_raw.best_params_['alpha'])

results['Model'].append('ElasticNet')
results['Target'].append('Log')
results['RMSE'].append(np.sqrt(mean_squared_error(y_test_log, best_elastic_log.predict(X_test))))
results['R²'].append(r2_score(y_test_log, best_elastic_log.predict(X_test)))
results['Best Alpha'].append(grid_elastic_log.best_params_['alpha'])

# --- Convertir resultados a DataFrame ---
results_df = pd.DataFrame(results)
print("=== Tabla Comparativa de Modelos ===")
print(results_df)

# --- Visualización de Predicciones (Log Target) ---
plt.figure(figsize=(16, 10))

plt.subplot(2, 2, 1)
plt.scatter(y_test_log, best_lasso_log.predict(X_test), alpha=0.6)
plt.plot([y_test_log.min(), y_test_log.max()], [y_test_log.min(), y_test_log.max()], 'r--')
plt.title(f'Lasso Log (RMSE: {results_df[(results_df["Model"] == "Lasso") & (results_df["Target"] == "Log")]["RMSE"].values[0]:.2f})')

plt.subplot(2, 2, 2)
plt.scatter(y_test_log, best_ridge_log.predict(X_test), alpha=0.6)
plt.plot([y_test_log.min(), y_test_log.max()], [y_test_log.min(), y_test_log.max()], 'r--')
plt.title(f'Ridge Log (RMSE: {results_df[(results_df["Model"] == "Ridge") & (results_df["Target"] == "Log")]["RMSE"].values[0]:.2f})')

plt.subplot(2, 2, 3)
plt.scatter(y_test_log, best_elastic_log.predict(X_test), alpha=0.6)
plt.plot([y_test_log.min(), y_test_log.max()], [y_test_log.min(), y_test_log.max()], 'r--')
plt.title(f'ElasticNet Log (RMSE: {results_df[(results_df["Model"] == "ElasticNet") & (results_df["Target"] == "Log")]["RMSE"].values[0]:.2f})')

plt.subplot(2, 2, 4)
plt.scatter(y_test_log, lr_log.predict(X_test), alpha=0.6)
plt.plot([y_test_log.min(), y_test_log.max()], [y_test_log.min(), y_test_log.max()], 'r--')
plt.title(f'Linear Log (RMSE: {results_df[(results_df["Model"] == "Linear") & (results_df["Target"] == "Log")]["RMSE"].values[0]:.2f})')

plt.tight_layout()
plt.show()

In [ ]:
!pip install statsmodels
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
import statsmodels.api as sm
from scipy import stats

# (Código de carga y preprocesamiento igual que antes)

# --- Identificar el mejor modelo (por RMSE) ---
best_model_name = results_df.loc[results_df['RMSE'].idxmin(), 'Model']
best_target = results_df.loc[results_df['RMSE'].idxmin(), 'Target']
print(f"🏆 Mejor modelo: {best_model_name} con target '{best_target}' (RMSE: {results_df['RMSE'].min():.2f})")

# --- Extraer el mejor modelo entrenado ---
if best_model_name == 'Linear':
    if best_target == 'Raw':
        best_model = lr_raw
        X_test_best = X_test
        y_test_best = y_test_raw
    else:
        best_model = lr_log
        X_test_best = X_test
        y_test_best = y_test_log
elif best_model_name == 'Lasso':
    if best_target == 'Raw':
        best_model = best_lasso_raw
    else:
        best_model = best_lasso_log
    X_test_best = X_test
    y_test_best = y_test_log
elif best_model_name == 'Ridge':
    if best_target == 'Raw':
        best_model = best_ridge_raw
    else:
        best_model = best_ridge_log
    X_test_best = X_test
    y_test_best = y_test_log
elif best_model_name == 'ElasticNet':
    if best_target == 'Raw':
        best_model = best_elastic_raw
    else:
        best_model = best_elastic_log
    X_test_best = X_test
    y_test_best = y_test_log

# --- 1. Análisis de Coeficientes ---
coef_df = pd.DataFrame({
    'Variable': X.columns,
    'Coeficiente': best_model.coef_
})
coef_df['Abs_Coef'] = np.abs(coef_df['Coeficiente'])
coef_df = coef_df.sort_values(by='Abs_Coef', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Coeficiente', y='Variable', data=coef_df.head(15), palette='viridis')
plt.title(f'Top 15 Coeficientes - {best_model_name} ({best_target})')
plt.tight_layout()
plt.show()

print("\n=== Coeficientes del Mejor Modelo ===")
print(coef_df.head(15))

# --- 2. Gráficos de Residuos ---
y_pred_best = best_model.predict(X_test_best)
residuals = y_test_best - y_pred_best

plt.figure(figsize=(15, 10))

# Histograma de residuos
plt.subplot(2, 2, 1)
sns.histplot(residuals, kde=True)
plt.title('Distribución de Residuos')

# Q-Q Plot (normalidad)
plt.subplot(2, 2, 2)
stats.probplot(residuals, dist="norm", plot=plt)
plt.title('Q-Q Plot (Normalidad)')

# Residuos vs Predichos (homocedasticidad)
plt.subplot(2, 2, 3)
plt.scatter(y_pred_best, residuals, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicciones')
plt.ylabel('Residuos')
plt.title('Residuos vs Predicciones (Homocedasticidad)')

# Residuos vs Orden (detección de patrones)
plt.subplot(2, 2, 4)
plt.plot(residuals, marker='o', linestyle='')
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Orden de Observación')
plt.ylabel('Residuos')
plt.title('Residuos vs Orden (Patrones)')

plt.tight_layout()
plt.show()

# --- 3. Estadísticas de Residuos ---
print("\n=== Estadísticas de Residuos ===")
print(f"Media de residuos: {np.mean(residuals):.4f} (debe ser cercana a 0)")
print(f"Desviación estándar: {np.std(residuals):.4f}")
print(f"Skewness: {stats.skew(residuals):.4f} (ideal: |valor| < 1)")
print(f"Kurtosis: {stats.kurtosis(residuals):.4f} (ideal: |valor| < 3)")

# --- 4. Diagnóstico de Normalidad (Shapiro-Wilk) ---
shapiro_stat, shapiro_p = stats.shapiro(residuals)
print(f"\nPrueba Shapiro-Wilk: p-value = {shapiro_p:.4f}")
if shapiro_p > 0.05:
    print("✅ Residuos siguen una distribución normal (p > 0.05)")
else:
    print("❌ Residuos NO siguen una distribución normal (p ≤ 0.05)")

In [ ]:
!pip install xgboost
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
from xgboost import plot_importance

# (Código de carga y preprocesamiento igual que antes)

# --- Modelo No Lineal: XGBoost ---
xgb_raw = xgb.XGBRegressor(random_state=42)
xgb_log = xgb.XGBRegressor(random_state=42)

# Hiperparámetros para GridSearchCV
param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

# GridSearch para XGBoost con target 'log_price' (generalmente mejor)
grid_xgb_log = GridSearchCV(xgb_log, param_grid_xgb, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid_xgb_log.fit(X_train, y_train_log)
best_xgb_log = grid_xgb_log.best_estimator_

# Predicciones
y_pred_xgb_log = best_xgb_log.predict(X_test)
rmse_xgb_log = np.sqrt(mean_squared_error(y_test_log, y_pred_xgb_log))
r2_xgb_log = r2_score(y_test_log, y_pred_xgb_log)

# --- Actualizar tabla de resultados ---
results_df = pd.DataFrame([
    {
        'Model': 'Lasso',
        'Target': 'Log',
        'RMSE': np.sqrt(mean_squared_error(y_test_log, best_lasso_log.predict(X_test))),
        'R²': r2_score(y_test_log, best_lasso_log.predict(X_test)),
        'Best Alpha': grid_lasso_log.best_params_['alpha']
    },
    {
        'Model': 'Ridge',
        'Target': 'Log',
        'RMSE': np.sqrt(mean_squared_error(y_test_log, best_ridge_log.predict(X_test))),
        'R²': r2_score(y_test_log, best_ridge_log.predict(X_test)),
        'Best Alpha': grid_ridge_log.best_params_['alpha']
    },
    {
        'Model': 'ElasticNet',
        'Target': 'Log',
        'RMSE': np.sqrt(mean_squared_error(y_test_log, best_elastic_log.predict(X_test))),
        'R²': r2_score(y_test_log, best_elastic_log.predict(X_test)),
        'Best Alpha': grid_elastic_log.best_params_['alpha']
    },
    {
        'Model': 'XGBoost',
        'Target': 'Log',
        'RMSE': rmse_xgb_log,
        'R²': r2_xgb_log,
        'Best Params': f"n_est={grid_xgb_log.best_params_['n_estimators']}, depth={grid_xgb_log.best_params_['max_depth']}"
    }
])

print("=== Tabla Comparativa (Target: Log) ===")
print(results_df)

# --- Visualización de Predicciones ---
plt.figure(figsize=(16, 10))

# Lasso
plt.subplot(2, 2, 1)
plt.scatter(y_test_log, best_lasso_log.predict(X_test), alpha=0.6)
plt.plot([y_test_log.min(), y_test_log.max()], [y_test_log.min(), y_test_log.max()], 'r--')
plt.title(f'Lasso Log (RMSE: {results_df.iloc[0]["RMSE"]:.2f})')

# Ridge
plt.subplot(2, 2, 2)
plt.scatter(y_test_log, best_ridge_log.predict(X_test), alpha=0.6)
plt.plot([y_test_log.min(), y_test_log.max()], [y_test_log.min(), y_test_log.max()], 'r--')
plt.title(f'Ridge Log (RMSE: {results_df.iloc[1]["RMSE"]:.2f})')

# ElasticNet
plt.subplot(2, 2, 3)
plt.scatter(y_test_log, best_elastic_log.predict(X_test), alpha=0.6)
plt.plot([y_test_log.min(), y_test_log.max()], [y_test_log.min(), y_test_log.max()], 'r--')
plt.title(f'ElasticNet Log (RMSE: {results_df.iloc[2]["RMSE"]:.2f})')

# XGBoost
plt.subplot(2, 2, 4)
plt.scatter(y_test_log, y_pred_xgb_log, alpha=0.6)
plt.plot([y_test_log.min(), y_test_log.max()], [y_test_log.min(), y_test_log.max()], 'r--')
plt.title(f'XGBoost Log (RMSE: {results_df.iloc[3]["RMSE"]:.2f})')

plt.tight_layout()
plt.show()

# --- Importancia de Variables en XGBoost ---
plt.figure(figsize=(10, 6))
plot_importance(best_xgb_log, max_num_features=15)
plt.title('Importancia de Variables - XGBoost (Log Target)')
plt.show()

# --- Residuos de XGBoost ---
residuals_xgb = y_test_log - y_pred_xgb_log

plt.figure(figsize=(15, 5))

# Histograma
plt.subplot(1, 3, 1)
sns.histplot(residuals_xgb, kde=True)
plt.title('Residuos XGBoost (Log)')

# Residuos vs Predichos
plt.subplot(1, 3, 2)
plt.scatter(y_pred_xgb_log, residuals_xgb, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicciones')
plt.ylabel('Residuos')
plt.title('Residuos vs Predicciones (XGBoost)')

# Q-Q Plot
plt.subplot(1, 3, 3)
stats.probplot(residuals_xgb, dist="norm", plot=plt)
plt.title('Q-Q Plot (XGBoost)')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import StackingRegressor
import xgboost as xgb

# (Código de carga y preprocesamiento igual que antes)

# --- Modelos Base (usados en el Stacking) ---
base_models = [
    ('lasso', Lasso(alpha=grid_lasso_log.best_params_['alpha'], random_state=42)),
    ('ridge', Ridge(alpha=grid_ridge_log.best_params_['alpha'], random_state=42)),
    ('elastic', ElasticNet(alpha=grid_elastic_log.best_params_['alpha'], random_state=42)),
    ('xgb', xgb.XGBRegressor(
        n_estimators=grid_xgb_log.best_params_['n_estimators'],
        max_depth=grid_xgb_log.best_params_['max_depth'],
        learning_rate=0.1,
        random_state=42
    ))
]

# --- Meta-Modelo (regresión lineal simple) ---
meta_model = LinearRegression()

# --- Crear Stacking Regressor ---
stacking_regressor = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model,
    cv=KFold(n_splits=5, shuffle=True, random_state=42)
)

# --- Entrenar Stacking ---
stacking_regressor.fit(X_train, y_train_log)
y_pred_stacking = stacking_regressor.predict(X_test)

# --- Métricas Stacking ---
rmse_stacking = np.sqrt(mean_squared_error(y_test_log, y_pred_stacking))
r2_stacking = r2_score(y_test_log, y_pred_stacking)

# --- Actualizar tabla de resultados ---
results_df = pd.DataFrame([
    {
        'Model': 'Lasso',
        'RMSE': np.sqrt(mean_squared_error(y_test_log, best_lasso_log.predict(X_test))),
        'R²': r2_score(y_test_log, best_lasso_log.predict(X_test))
    },
    {
        'Model': 'Ridge',
        'RMSE': np.sqrt(mean_squared_error(y_test_log, best_ridge_log.predict(X_test))),
        'R²': r2_score(y_test_log, best_ridge_log.predict(X_test))
    },
    {
        'Model': 'ElasticNet',
        'RMSE': np.sqrt(mean_squared_error(y_test_log, best_elastic_log.predict(X_test))),
        'R²': r2_score(y_test_log, best_elastic_log.predict(X_test))
    },
    {
        'Model': 'XGBoost',
        'RMSE': rmse_xgb_log,
        'R²': r2_xgb_log
    },
    {
        'Model': 'Stacking',
        'RMSE': rmse_stacking,
        'R²': r2_stacking
    }
])

print("=== Tabla Comparativa Final (Target: Log) ===")
print(results_df)

# --- Visualización de Predicciones Stacking ---
plt.figure(figsize=(16, 12))

# Stacking vs Real
plt.subplot(2, 2, 1)
plt.scatter(y_test_log, y_pred_stacking, alpha=0.6)
plt.plot([y_test_log.min(), y_test_log.max()], [y_test_log.min(), y_test_log.max()], 'r--')
plt.title(f'Stacking (RMSE: {rmse_stacking:.2f}, R²: {r2_stacking:.2f})')

# Comparación de todos los modelos
plt.subplot(2, 2, 2)
plt.scatter(y_test_log, best_lasso_log.predict(X_test), alpha=0.6, label='Lasso')
plt.scatter(y_test_log, best_ridge_log.predict(X_test), alpha=0.6, label='Ridge')
plt.scatter(y_test_log, best_elastic_log.predict(X_test), alpha=0.6, label='ElasticNet')
plt.scatter(y_test_log, y_pred_xgb_log, alpha=0.6, label='XGBoost')
plt.scatter(y_test_log, y_pred_stacking, alpha=0.6, label='Stacking', marker='x')
plt.plot([y_test_log.min(), y_test_log.max()], [y_test_log.min(), y_test_log.max()], 'r--')
plt.legend()
plt.title('Comparación de Todos los Modelos')

# Residuos Stacking
residuals_stacking = y_test_log - y_pred_stacking
plt.subplot(2, 2, 3)
plt.scatter(y_pred_stacking, residuals_stacking, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicciones')
plt.ylabel('Residuos')
plt.title('Residuos vs Predicciones (Stacking)')

# Histograma de residuos Stacking
plt.subplot(2, 2, 4)
sns.histplot(residuals_stacking, kde=True)
plt.title('Distribución de Residuos (Stacking)')

plt.tight_layout()
plt.show()

# --- Análisis del Meta-Modelo ---
print("\n=== Coeficientes del Meta-Modelo (LinearRegression) ===")
meta_coefs = pd.DataFrame({
    'Base Model': [name for name, _ in base_models],
    'Weight': stacking_regressor.final_estimator_.coef_
})
print(meta_coefs)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import StackingRegressor, VotingRegressor
import xgboost as xgb

# (Código de carga y preprocesamiento igual que antes)

# --- Modelos Base (voting) ---
voting_models = [
    ('lasso', Lasso(alpha=grid_lasso_log.best_params_['alpha'], random_state=42)),
    ('ridge', Ridge(alpha=grid_ridge_log.best_params_['alpha'], random_state=42)),
    ('elastic', ElasticNet(alpha=grid_elastic_log.best_params_['alpha'], random_state=42)),
    ('xgb', xgb.XGBRegressor(
        n_estimators=grid_xgb_log.best_params_['n_estimators'],
        max_depth=grid_xgb_log.best_params_['max_depth'],
        learning_rate=0.1,
        random_state=42
    ))
]

# --- Voting Regressor (promedio simple) ---
voting_regressor = VotingRegressor(estimators=voting_models)
voting_regressor.fit(X_train, y_train_log)
y_pred_voting = voting_regressor.predict(X_test)

# --- Métricas Voting ---
rmse_voting = np.sqrt(mean_squared_error(y_test_log, y_pred_voting))
r2_voting = r2_score(y_test_log, y_pred_voting)

# --- Actualizar tabla de resultados ---
results_df = pd.DataFrame([
    {
        'Model': 'Lasso',
        'RMSE': np.sqrt(mean_squared_error(y_test_log, best_lasso_log.predict(X_test))),
        'R²': r2_score(y_test_log, best_lasso_log.predict(X_test))
    },
    {
        'Model': 'Ridge',
        'RMSE': np.sqrt(mean_squared_error(y_test_log, best_ridge_log.predict(X_test))),
        'R²': r2_score(y_test_log, best_ridge_log.predict(X_test))
    },
    {
        'Model': 'ElasticNet',
        'RMSE': np.sqrt(mean_squared_error(y_test_log, best_elastic_log.predict(X_test))),
        'R²': r2_score(y_test_log, best_elastic_log.predict(X_test))
    },
    {
        'Model': 'XGBoost',
        'RMSE': rmse_xgb_log,
        'R²': r2_xgb_log
    },
    {
        'Model': 'Voting',
        'RMSE': rmse_voting,
        'R²': r2_voting
    },
    {
        'Model': 'Stacking',
        'RMSE': rmse_stacking,
        'R²': r2_stacking
    }
])

print("=== Tabla Comparativa Final (Target: Log) ===")
print(results_df)

# --- Visualización de Predicciones ---
plt.figure(figsize=(18, 10))

# Voting vs Real
plt.subplot(2, 3, 1)
plt.scatter(y_test_log, y_pred_voting, alpha=0.6)
plt.plot([y_test_log.min(), y_test_log.max()], [y_test_log.min(), y_test_log.max()], 'r--')
plt.title(f'Voting (RMSE: {rmse_voting:.2f}, R²: {r2_voting:.2f})')

# Stacking vs Real
plt.subplot(2, 3, 2)
plt.scatter(y_test_log, y_pred_stacking, alpha=0.6)
plt.plot([y_test_log.min(), y_test_log.max()], [y_test_log.min(), y_test_log.max()], 'r--')
plt.title(f'Stacking (RMSE: {rmse_stacking:.2f}, R²: {r2_stacking:.2f})')

# Comparación de todos los modelos
plt.subplot(2, 3, 3)
plt.scatter(y_test_log, best_lasso_log.predict(X_test), alpha=0.6, label='Lasso')
plt.scatter(y_test_log, best_ridge_log.predict(X_test), alpha=0.6, label='Ridge')
plt.scatter(y_test_log, best_elastic_log.predict(X_test), alpha=0.6, label='ElasticNet')
plt.scatter(y_test_log, y_pred_xgb_log, alpha=0.6, label='XGBoost')
plt.scatter(y_test_log, y_pred_voting, alpha=0.6, label='Voting', marker='x')
plt.scatter(y_test_log, y_pred_stacking, alpha=0.6, label='Stacking', marker='o')
plt.plot([y_test_log.min(), y_test_log.max()], [y_test_log.min(), y_test_log.max()], 'r--')
plt.legend()
plt.title('Todos los Modelos')

# Residuos Voting
residuals_voting = y_test_log - y_pred_voting
plt.subplot(2, 3, 4)
plt.scatter(y_pred_voting, residuals_voting, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicciones')
plt.ylabel('Residuos')
plt.title('Residuos vs Predicciones (Voting)')

# Residuos Stacking
residuals_stacking = y_test_log - y_pred_stacking
plt.subplot(2, 3, 5)
plt.scatter(y_pred_stacking, residuals_stacking, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicciones')
plt.ylabel('Residuos')
plt.title('Residuos vs Predicciones (Stacking)')

# Histograma de residuos (Voting vs Stacking)
plt.subplot(2, 3, 6)
sns.histplot(residuals_voting, kde=True, label='Voting', color='blue')
sns.histplot(residuals_stacking, kde=True, label='Stacking', color='red')
plt.legend()
plt.title('Distribución de Residuos')

plt.tight_layout()
plt.show()

# --- Análisis de Consistencia entre Voting y Stacking ---
print("\n=== Correlación entre Voting y Stacking ===")
correlation = np.corrcoef(y_pred_voting, y_pred_stacking)[0, 1]
print(f"Correlación: {correlation:.4f}")

# --- RMSE Promedio de Modelos Individuales ---
individual_rmse = np.mean([
    np.sqrt(mean_squared_error(y_test_log, best_lasso_log.predict(X_test))),
    np.sqrt(mean_squared_error(y_test_log, best_ridge_log.predict(X_test))),
    np.sqrt(mean_squared_error(y_test_log, best_elastic_log.predict(X_test))),
    rmse_xgb_log
])
print(f"\nRMSE Promedio Modelos Individuales: {individual_rmse:.4f}")
print(f"RMSE Voting: {rmse_voting:.4f}")
print(f"RMSE Stacking: {rmse_stacking:.4f}")